# FOM Weight Calibration — Exploratory Analysis (issue #3188)

Companion notebook to `scripts/research/calibrate_fom.py` and `docs/research/fom_calibration.md`.

This notebook loads the cached term-vectors and per-board weights produced by the calibration script and provides interactive exploration:

1. Per-board term distributions (committed vs perturbations)
2. Weight sensitivity analysis (how rank_consistency changes around the selected weights)
3. Train vs holdout breakdown
4. Per-term informativeness across boards

In [ ]:
import json
from pathlib import Path

import numpy as np

from kicad_tools.optim.fom import SOFT_TERM_NAMES

OUTPUT_DIR = Path("../data/research/fom_weights")
REPORT = json.loads((OUTPUT_DIR / "calibration_report.json").read_text())
CACHE = np.load(OUTPUT_DIR / "term_cache.npz")
BOARDS = sorted({k.rsplit("__", 1)[0] for k in CACHE.files})
print(f"Boards in cache: {BOARDS}")
print(f"Pareto frontier size: {REPORT['pareto_summary']['pareto_size']}")

## 1. Per-board term distributions

For each board, plot the per-term value distribution across 40 perturbations and mark the committed value. Terms where the committed marker is at or past the right tail are informative; terms where it sits in the middle of the distribution carry no signal for that board.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(BOARDS), len(SOFT_TERM_NAMES), figsize=(20, 14), sharex=False)
for bi, board in enumerate(BOARDS):
    c = CACHE[f"{board}__committed"]
    p = CACHE[f"{board}__perturbed"]
    for ti, term in enumerate(SOFT_TERM_NAMES):
        ax = axes[bi, ti]
        ax.hist(p[:, ti], bins=10, alpha=0.6, color="steelblue")
        ax.axvline(c[ti], color="crimson", linewidth=2)
        if bi == 0:
            ax.set_title(term[:12], fontsize=8)
        if ti == 0:
            ax.set_ylabel(board[:8], fontsize=8)
        ax.tick_params(labelsize=6)
plt.suptitle("Per-board term distributions (red = committed)", y=1.005)
plt.tight_layout()
plt.show()

## 2. Per-term rank improvement vs uniform

How much does each board's rank_consistency improve under the calibrated weights vs uniform=1.0?

In [ ]:
train = REPORT["train_metrics"]
holdout = REPORT["holdout_metrics"]
u_train = REPORT["uniform_train_metrics"]
u_holdout = REPORT["uniform_holdout_metrics"]

labels = list(train.keys()) + list(holdout.keys())
tuned = [train.get(b, holdout.get(b))["rank_consistency"] for b in labels]
uniform = [u_train.get(b, u_holdout.get(b))["rank_consistency"] for b in labels]
is_holdout = [b in holdout for b in labels]

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - 0.2, uniform, 0.4, label="uniform=1.0", color="lightgray")
ax.bar(x + 0.2, tuned, 0.4, label="tuned", color=["crimson" if h else "steelblue" for h in is_holdout])
ax.axhline(0.7, color="black", linestyle="--", alpha=0.4, label="AC #3 (0.7)")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylabel("rank consistency")
ax.set_title("Tuned vs uniform weights (red = holdout boards)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Selected global weights

What weights did the Pareto sweep end up choosing as the global default?

In [ ]:
global_w = REPORT["global_weights"]
weight_pairs = sorted(global_w.items(), key=lambda x: -x[1])
names = [w[0] for w in weight_pairs]
values = [w[1] for w in weight_pairs]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(names[::-1], values[::-1], color="steelblue")
ax.set_xscale("log")
ax.set_xlabel("weight (log scale)")
ax.set_title("Calibrated global FOM weights (issue #3188 Pareto sweep)")
plt.tight_layout()
plt.show()
print("Selected geo-mean rank_consistency across train boards:", REPORT["pareto_summary"]["selected_geo_mean_rc"])